# 02 · Chạy thử code chính thức: Stage 1 (VLM) + Stage 2 (SASRec) — runtime **GPU**

Nhánh `explore/official-code`. Cần chạy notebook 01 trước (dữ liệu nằm trên Drive).

* **Runtime:** `Runtime → Change runtime type → GPU`, chọn **A100** (khuyến nghị) hoặc **L4**. **T4 không dùng được** vì không có bf16.
* Các phần (thời gian là **ước tính**, chưa đo thực tế):

| Phần | Nội dung | Ước tính (A100) |
|---|---|---|
| 0 | Setup, cài thư viện đúng phiên bản tác giả | 5–10 phút |
| A | Probe hành vi thực tế của code (prompt, token, loss, eval) | 2–5 phút |
| B | SASRec chỉ ID (Stage 2 chính thức) → so hàng SASRec của paper | 5–15 phút/seed |
| C | VLM_Vanilla (không train): bảng item + full ranking + plug | 30–50 phút |
| D | **Smoke** VLM2Rec (K_u=256 user, 1 epoch) → kiểm tra pipeline chạy trọn | 20–40 phút |
| E | (tuỳ chọn) **Train đầy đủ** VLM2Rec 3 epoch → so với paper | ~2–4 giờ |
| F | Tổng hợp bảng so sánh với paper → `summary_<DS>.md` | 1 phút |

Mọi kết quả JSON lưu ở `MyDrive/VLM2Rec_work/audit_results/`.

In [ ]:
REPO = "TrNguyenMQuan/VLM2Rec"
BRANCH = "explore/official-code"
DATASET = "Beauty"
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
DRIVE_ROOT = "/content/drive/MyDrive/VLM2Rec_work"

STAGE2_SEEDS = "42"            # paper dùng 5 seed: "42 43 44 45 46"
EVAL_MAX_USERS = 0             # 0 = toàn bộ user test (so được với paper); 5000 để tiết kiệm thời gian
SMOKE_K, SMOKE_EPOCHS, SMOKE_EVAL_USERS = 256, 1, 3000
RUN_FULL = False               # True để train đầy đủ (Phần E, vài giờ)
FULL_EPOCHS, FULL_BATCH = 3, 8 # batch mặc định của code = 8; paper ghi 32 (thử 32 nếu GPU đủ bộ nhớ)
RUN_OFFICIAL_SAMPLED_EVAL = False  # chạy thêm evaluate.py gốc (100 negative ngẫu nhiên) — chậm

## Phần 0 · Setup

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Chưa có GPU: Runtime → Change runtime type → GPU (A100/L4)"
assert torch.cuda.is_bf16_supported(), "GPU này không hỗ trợ bf16 (vd. T4) → đổi sang A100 hoặc L4"
print("torch", torch.__version__, "| bf16 OK")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, subprocess

def clone_repo():
    """Public repo: clone directly. Private repo: add a Colab secret GITHUB_TOKEN (read access)."""
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    dst = "/content/VLM2Rec"
    if os.path.exists(f"{dst}/.git"):
        subprocess.run(["git", "-C", dst, "pull", "--ff-only"], check=True)
    else:
        url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", url, dst], check=True)
    print(subprocess.run(["git", "-C", dst, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

clone_repo()
TOOLS = "/content/VLM2Rec/tools/official"
WORK = "/content/work/official"          # working copy of the official code (repo copy stays untouched)
if not os.path.exists(WORK):
    shutil.copytree("/content/VLM2Rec/third_party/vlm2rec_anon", WORK)
AUDIT = f"{DRIVE_ROOT}/audit_results"
for p in (DRIVE_ROOT, AUDIT, f"{DRIVE_ROOT}/data", f"{DRIVE_ROOT}/checkpoints", f"{DRIVE_ROOT}/logs"):
    os.makedirs(p, exist_ok=True)
print("WORK =", WORK, "| AUDIT =", AUDIT)

# khôi phục dữ liệu do notebook 01 tạo
tarball = f"{DRIVE_ROOT}/data/{DATASET}_processed.tar.gz"
if not os.path.exists(f"{WORK}/datasets/processed/{DATASET}.txt"):
    assert os.path.exists(tarball), f"Không thấy {tarball} — hãy chạy notebook 01 trước"
    os.makedirs(f"{WORK}/datasets", exist_ok=True)
    subprocess.run(["tar", "xzf", tarball, "-C", f"{WORK}/datasets"], check=True)
print(sorted(os.listdir(f"{WORK}/datasets/processed"))[:10])
CKPT, LOGS = f"{DRIVE_ROOT}/checkpoints", f"{DRIVE_ROOT}/logs"
MTAG = MODEL.replace("/", "_")

def persist():
    """Chép embedding item (results/) lên Drive để không mất khi runtime ngắt."""
    if os.path.exists(f"{WORK}/results"):
        shutil.copytree(f"{WORK}/results", f"{DRIVE_ROOT}/results", dirs_exist_ok=True)

def official_paths(id_time, epochs, temp_cf=0.05, temp_aux=0.1, lmd=0.1, seed=42):
    """Đường dẫn adapter LoRA và bảng item mà main.py của tác giả ghi ra."""
    desc = f"wpcl_{DATASET}_._{id_time}"
    adapter = f"{CKPT}/{DATASET}/{desc}/{MTAG}/seed_{seed}/epoch_{epochs}/cf_temp_{temp_cf}/aux_temp_{temp_aux}/lmd_{lmd}/hf_model"
    emb = f"{WORK}/results/vlm_item_emb/{DATASET}/{desc}/epoch_{epochs}/{MTAG}"
    for p in (adapter, emb):
        print(("OK      " if os.path.exists(p) else "MISSING ") + p)
    return adapter, emb

In [ ]:
# Phiên bản tác giả đã test: torch 2.7, transformers 4.55, peft 0.17. qwen-vl-utils phải là 0.0.14 (có image_patch_size).
# accelerate không có trong requirements.txt của tác giả nhưng cần cho device_map="auto".
!pip install -q "transformers==4.55.0" "peft==0.17.0" "qwen-vl-utils==0.0.14" accelerate "pandas>=2.2,<3"
!python -c "import torch, transformers, peft, accelerate; from importlib.metadata import version; print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '| accelerate', accelerate.__version__, '| qwen-vl-utils', version('qwen-vl-utils'))"
# (Tuỳ chọn, nếu gặp lỗi tương thích) khớp torch đúng bản tác giả rồi Runtime → Restart:
# !pip install -q torch==2.7.0 torchvision==0.22.0 --index-url https://download.pytorch.org/whl/cu126

## Phần A · Probe hành vi thực tế
Xác nhận bằng runtime những gì review tĩnh đã phát hiện: phiên bản và requirements, chuỗi prompt, số token/ảnh, β/τ của WPCL có được học không, ý nghĩa thật của τ trong CRTR, giao thức eval, mặc định của Stage 2.
Trạng thái: `OK` = đúng kỳ vọng · `DIFF` = khác paper (là phát hiện) · `WARN` = có vấn đề · `INFO` = để tham khảo.

In [ ]:
!python {TOOLS}/audit_probes.py --official_dir {WORK} --data_name {DATASET} --model_name {MODEL} --out {AUDIT}/probes_{DATASET}.json

## Phần B · SASRec chỉ ID (Stage 2 chính thức)
Dùng dữ liệu `tail10` của code gốc. Nếu số **khớp** hàng SASRec của paper (Beauty N@20 = 0.0483), thì paper đã dùng dữ liệu cắt 10 item (quyết định D-01).

In [ ]:
print('Stage 2 đang chạy (log được rút gọn, mỗi seed ~5–15 phút)...')
!python {TOOLS}/run_stage2.py --official_dir {WORK} --data_name {DATASET} --mode id_only --label SASRec_ID --paper_key sasrec --seeds {STAGE2_SEEDS} --out_json {AUDIT}/stage2_{DATASET}.json 2>&1 | tail -n 25

## Phần C · VLM_Vanilla (không train)
Phép thử rẻ nhất cho độ trung thực của prompt, pooling và đánh giá. `fullrank_eval.py` dùng `generate_item_embeddings` của tác giả với model gốc, rồi xếp hạng **toàn bộ item** (paper: Beauty N@20 = 0.0442). Sau đó cắm bảng item vào SASRec theo 2 cách:
* `modality_only` = chỉ `W·e_i` (đúng như paper mô tả; paper: 0.0463, thấp hơn SASRec ID)
* `concat` = `[ID ‖ W·e_i]` (mặc định của code)

In [ ]:
!python {TOOLS}/fullrank_eval.py --official_dir {WORK} --data_name {DATASET} --model_name {MODEL} --tag vanilla --paper_key vanilla --max_users {EVAL_MAX_USERS} --geometry --out_json {AUDIT}/fullrank_{DATASET}_vanilla.json 2>&1 | tee {LOGS}/fullrank_vanilla.log
persist()

In [ ]:
VAN_EMB = f"{WORK}/results/vlm_item_emb/{DATASET}/vanilla/epoch_0/{MTAG}"
!python {TOOLS}/run_stage2.py --official_dir {WORK} --data_name {DATASET} --mode modality_only --emb_dir {VAN_EMB} --emb_name vanilla --label VLM_Vanilla_plug_modality_only --paper_key vanilla_plug --seeds {STAGE2_SEEDS} --out_json {AUDIT}/stage2_{DATASET}.json 2>&1 | tail -n 8
!python {TOOLS}/run_stage2.py --official_dir {WORK} --data_name {DATASET} --mode concat --emb_dir {VAN_EMB} --emb_name vanilla --label VLM_Vanilla_plug_concat --paper_key vanilla_plug --seeds {STAGE2_SEEDS} --out_json {AUDIT}/stage2_{DATASET}.json 2>&1 | tail -n 8

## Phần D · Smoke test VLM2Rec (đúng lệnh trong README của tác giả, thu nhỏ)
`main.py` của tác giả với `--data_sample_K 256 --epochs 1 --phase embed`: train WPCL+CRTR trên 256 user → lưu adapter → sinh bảng item. Mục đích là **kiểm tra pipeline chạy trọn**, chưa phải để so số. Adapter ghi thẳng lên Drive (`--save_root`).

In [ ]:
SMOKE_ID = "smoke"
cmd = (f"cd {WORK} && python main.py --model_name {MODEL} --data_name {DATASET} --data_path ./datasets/processed "
       f"--loss_type wpcl --aux_loss crtr --temp_cf 0.05 --temp_aux 0.1 --lmd 0.1 --epochs {SMOKE_EPOCHS} "
       f"--batch_size 8 --lr 1e-5 --data_sample_K {SMOKE_K} --gpu_id 0 --id_time {SMOKE_ID} --save_root {CKPT}/ --phase embed")
print(cmd)
!{cmd} 2>&1 | tee {LOGS}/stage1_{SMOKE_ID}.log
persist()
SMOKE_ADAPTER, SMOKE_EMB = official_paths(SMOKE_ID, SMOKE_EPOCHS)

In [ ]:
!python {TOOLS}/fullrank_eval.py --official_dir {WORK} --data_name {DATASET} --model_name {MODEL} --adapter_path {SMOKE_ADAPTER} --item_emb_dir {SMOKE_EMB} --tag smoke --paper_key none --max_users {SMOKE_EVAL_USERS} --out_json {AUDIT}/fullrank_{DATASET}_smoke.json
!python {TOOLS}/run_stage2.py --official_dir {WORK} --data_name {DATASET} --mode modality_only --emb_dir {SMOKE_EMB} --emb_name vlm2rec_smoke --label Smoke_plug_modality_only --paper_key none --seeds 42 --out_json {AUDIT}/stage2_{DATASET}.json 2>&1 | tail -n 8

In [ ]:
# (Tuỳ chọn) đánh giá gốc của tác giả: target so với 100 negative ngẫu nhiên — để thấy con số mà code gốc in ra
if RUN_OFFICIAL_SAMPLED_EVAL:
    cmd = (f"cd {WORK} && python evaluate.py --model_name {MODEL} --data_name {DATASET} --data_path ./datasets/processed "
           f"--epochs {SMOKE_EPOCHS} --temp_cf 0.05 --temp_aux 0.1 --lmd 0.1 --id_time {SMOKE_ID} --save_root {CKPT}/ "
           f"--phase eval --eval_batch_size 8")
    !{cmd} 2>&1 | tail -n 15

## Phần E · (Tuỳ chọn) Train đầy đủ VLM2Rec — so sánh thật với paper
Đặt `RUN_FULL = True` ở cell cấu hình rồi chạy. Cấu hình đúng như README của tác giả (3 epoch, lr 1e-5, τ_WPCL 0.05, τ_CRTR 0.1, λ 0.1; batch = `FULL_BATCH`).
⚠️ Trainer của tác giả **không có resume**, chỉ lưu checkpoint ở epoch 1 và epoch cuối. Hãy giữ tab mở, hoặc dùng Colab Pro+ chạy nền.
Paper (Beauty): VLM2Rec N@20 = 0.0784 (full ranking), VLM2Rec_plug N@20 = 0.0728.

In [ ]:
if RUN_FULL:
    FULL_ID = "full"
    cmd = (f"cd {WORK} && python main.py --model_name {MODEL} --data_name {DATASET} --data_path ./datasets/processed "
           f"--loss_type wpcl --aux_loss crtr --temp_cf 0.05 --temp_aux 0.1 --lmd 0.1 --epochs {FULL_EPOCHS} "
           f"--batch_size {FULL_BATCH} --lr 1e-5 --data_sample_K 0 --gpu_id 0 --id_time {FULL_ID} --save_root {CKPT}/ --phase embed")
    print(cmd)
    !{cmd} 2>&1 | tee {LOGS}/stage1_{FULL_ID}.log
    persist()
    FULL_ADAPTER, FULL_EMB = official_paths(FULL_ID, FULL_EPOCHS)

In [ ]:
if RUN_FULL:
    !python {TOOLS}/fullrank_eval.py --official_dir {WORK} --data_name {DATASET} --model_name {MODEL} --adapter_path {FULL_ADAPTER} --item_emb_dir {FULL_EMB} --tag full --paper_key vlm2rec --max_users {EVAL_MAX_USERS} --geometry --out_json {AUDIT}/fullrank_{DATASET}_full.json
    !python {TOOLS}/run_stage2.py --official_dir {WORK} --data_name {DATASET} --mode modality_only --emb_dir {FULL_EMB} --emb_name vlm2rec_full --label VLM2Rec_plug_modality_only --paper_key vlm2rec_plug --seeds {STAGE2_SEEDS} --out_json {AUDIT}/stage2_{DATASET}.json 2>&1 | tail -n 8
    !python {TOOLS}/run_stage2.py --official_dir {WORK} --data_name {DATASET} --mode concat --emb_dir {FULL_EMB} --emb_name vlm2rec_full --label VLM2Rec_plug_concat --paper_key vlm2rec_plug --seeds {STAGE2_SEEDS} --out_json {AUDIT}/stage2_{DATASET}.json 2>&1 | tail -n 8

## Phần F · Tổng hợp so với paper

In [ ]:
!python {TOOLS}/summarize.py --audit_dir {AUDIT} --data_name {DATASET} --out {AUDIT}/summary_{DATASET}.md > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"{AUDIT}/summary_{DATASET}.md").read()))

### Đưa kết quả về repo
1. Tải thư mục `MyDrive/VLM2Rec_work/audit_results/` về máy (các file `*.json` và `summary_*.md`; **không** tải checkpoint).
2. Chép vào `docs/official_code/results/` trên nhánh `explore/official-code`, rồi commit.
3. Điền bảng trong `docs/official_code/AUDIT.md` (hoặc nhờ Claude: *"đọc docs/official_code/results và cập nhật AUDIT.md"*).